In [11]:
import os
#os.chdir("C:\\Users\\Le Tian\\Desktop\\Ensemble modeling\\comp-401")
os.chdir("/Users/gaoletian/Desktop/Ensemble modeling/comp-401")
os.getcwd()

'/Users/gaoletian/Desktop/Ensemble modeling/comp-401'

In [13]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from multi_omics_integration.processing import load_resize
from multi_omics_integration.func import estimators, estimator_parameters

In [14]:
datasets = {
    "methylation": "../Datasets/kipan/staging/Methylation.csv"
}
labels = "../Datasets/kipan/staging/Clinical_enc.csv"
target_name = "stage_enc"

merged_X, X_modalities, y = load_resize(datasets, labels, target_name)

In [17]:
import scanpy as sc
import pandas as pd

# Load data
meth = pd.read_csv(datasets['methylation'], index_col=0)

# Methylation
n_meth = meth.fillna(meth.mean())

from sklearn.feature_selection import VarianceThreshold

def remove_low_variance(X, threshold=0.01):
    selector = VarianceThreshold(threshold=threshold)
    X_filtered = selector.fit_transform(X)
    selected_features = X.columns[selector.get_support()]
    return pd.DataFrame(X_filtered, columns=selected_features)

meth_df = remove_low_variance(n_meth)

print("Processed Methylation shape:", meth_df.shape)

Processed Methylation shape: (558, 9734)


In [18]:
import pandas as pd

# Convert y to a pandas Series with same index as rna_processed
y = pd.Series(y, index=meth_df.index)

# Now reassign
merged_X = meth_df.loc[y.index]

print(" Merged shape:", merged_X.shape)
print(" Target distribution:\n", y.value_counts())


 Merged shape: (558, 9734)
 Target distribution:
 0    350
1    208
Name: count, dtype: int64


In [21]:
results = []

for name, model in estimators:
    if name not in estimator_parameters:
        print(f"Skipping '{name}' (no param grid defined)")
        continue

    print(f"\nTuning model: {name}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])

    param_grid = {f"clf__{k}": v for k, v in estimator_parameters[name].items()}

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        verbose=1,
        n_jobs=-1
    )

    try:
        grid.fit(merged_X, y)
        results.append({
            "model": name,
            "best_score": grid.best_score_,
            "best_params": grid.best_params_
        })
        print(f" Best score for {name}: {grid.best_score_:.4f}")
        print(f" Best parameters for {name}: {grid.best_params_}")
    except Exception as e:
        print(f" Failed to fit {name}: {e}")


Tuning model: logistic
Fitting 5 folds for each of 5 candidates, totalling 25 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_c

 Best score for logistic: 0.7636
 Best parameters for logistic: {'clf__C': 0.01}

Tuning model: random_forest
Fitting 5 folds for each of 16 candidates, totalling 80 fits
 Best score for random_forest: 0.7564
 Best parameters for random_forest: {'clf__max_depth': 5, 'clf__n_estimators': 200}

Tuning model: deep_nn
Fitting 5 folds for each of 864 candidates, totalling 4320 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

 Best score for deep_nn: 0.7689
 Best parameters for deep_nn: {'clf__activation': 'tanh', 'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (100,), 'clf__learning_rate': 'constant', 'clf__solver': 'sgd'}
Skipping 'svc' (no param grid defined)


In [22]:
results_df = pd.DataFrame(results).sort_values(by="best_score", ascending=False)
print("\nSummary of Results:")
display(results_df)


Summary of Results:


,model,best_score,best_params
2,deep_nn,0.768903,"{'clf__activation': 'tanh', 'clf__alpha': 0.00..."
0,logistic,0.763626,{'clf__C': 0.01}
1,random_forest,0.756355,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
